In [1]:
from pwn import *
import os

filepath = os.path.abspath("handout/chal")
elf = ELF(filepath)

[*] '/workspaces/alpaca-ctf/2026-03/21-daily-alpacker/handout/chal'
    Arch:       amd64-64-little
    RELRO:      Full RELRO
    Stack:      Canary found
    NX:         NX enabled
    PIE:        PIE enabled
    SHSTK:      Enabled
    IBT:        Enabled


In [2]:
hex(elf.entrypoint)

'0x1140'

In [3]:
# Sections
print("addr\tsize\tname")
for s in sorted(elf.sections, key=lambda s: s.header.sh_addr):
    h = s.header
    print(f"{h.sh_addr:#x}\t{h.sh_size:#x}\t{s.name}")

addr	size	name
0x0	0x0	
0x0	0x2d	.comment
0x0	0x10a	.shstrtab
0x318	0x1c	.interp
0x338	0x30	.note.gnu.property
0x368	0x24	.note.gnu.build-id
0x38c	0x20	.note.ABI-tag
0x3b0	0x28	.gnu.hash
0x3d8	0x168	.dynsym
0x540	0xe0	.dynstr
0x620	0x1e	.gnu.version
0x640	0x50	.gnu.version_r
0x690	0xd8	.rela.dyn
0x768	0xc0	.rela.plt
0x1000	0x1b	.init
0x1020	0x90	.plt
0x10b0	0x10	.plt.got
0x10c0	0x80	.plt.sec
0x1140	0x2cb	.text
0x140c	0xd	.fini
0x2000	0x1d	.rodata
0x2020	0x34	.eh_frame_hdr
0x2058	0xac	.eh_frame
0x3d80	0x8	.init_array
0x3d88	0x8	.fini_array
0x3d90	0x1f0	.dynamic
0x3f80	0x80	.got
0x4000	0x13b	.data
0x4140	0x10	.bss


In [4]:
print(disasm(elf.section(".text")))

   0:   f3 0f 1e fa             endbr64
   4:   31 ed                   xor    ebp, ebp
   6:   49                      dec    ecx
   7:   89 d1                   mov    ecx, edx
   9:   5e                      pop    esi
   a:   48                      dec    eax
   b:   89 e2                   mov    edx, esp
   d:   48                      dec    eax
   e:   83 e4 f0                and    esp, 0xfffffff0
  11:   50                      push   eax
  12:   54                      push   esp
  13:   45                      inc    ebp
  14:   31 c0                   xor    eax, eax
  16:   31 c9                   xor    ecx, ecx
  18:   48                      dec    eax
  19:   8d 3d ca 00 00 00       lea    edi, ds:0xca
  1f:   ff 15 73 2e 00 00       call   DWORD PTR ds:0x2e73
  25:   f4                      hlt
  26:   66 2e 0f 1f 84 00 00 00 00 00   nop    WORD PTR cs:[eax+eax*1+0x0]
  30:   48                      dec    eax
  31:   8d 3d c9 2f 00 00       lea    edi, ds:0x2fc9
  

In [5]:
prog = elf.section(".data")[0x20 : 0x20 + 0x11b]
prog_decoded = bytes([(b * ord('s')) % 256 for b in prog])
print(disasm(prog_decoded))

   0:   31 c0                   xor    eax, eax
   2:   80 3f 41                cmp    BYTE PTR [edi], 0x41
   5:   0f 85 0f 01 00 00       jne    0x11a
   b:   80 7f 23 7d             cmp    BYTE PTR [edi+0x23], 0x7d
   f:   0f 85 05 01 00 00       jne    0x11a
  15:   80 7f 01 6c             cmp    BYTE PTR [edi+0x1], 0x6c
  19:   0f 85 fb 00 00 00       jne    0x11a
  1f:   80 7f 10 6e             cmp    BYTE PTR [edi+0x10], 0x6e
  23:   0f 85 f1 00 00 00       jne    0x11a
  29:   80 7f 04 63             cmp    BYTE PTR [edi+0x4], 0x63
  2d:   0f 85 e7 00 00 00       jne    0x11a
  33:   80 7f 14 39             cmp    BYTE PTR [edi+0x14], 0x39
  37:   0f 85 dd 00 00 00       jne    0x11a
  3d:   80 7f 05 61             cmp    BYTE PTR [edi+0x5], 0x61
  41:   0f 85 d3 00 00 00       jne    0x11a
  47:   80 7f 08 61             cmp    BYTE PTR [edi+0x8], 0x61
  4b:   0f 85 c9 00 00 00       jne    0x11a
  51:   80 7f 17 61             cmp    BYTE PTR [edi+0x17], 0x61
  55:   0f 85 bf

In [6]:
prog_hex = prog_decoded.hex(" ")

In [7]:
buf = [0] * 0x24
buf[0] = int(re.search(r"80 3f (..)", prog_hex).group(1), 16)
for addr, val in re.findall(r"80 7f (..) (..)", prog_hex):
    buf[int(addr, 16)] = int(val, 16)

In [ ]:
bytes(buf)